# Notebook 1: Introduction to Hyperscanning Analysis with HyPyP

In this notebook, we introduce the basics of hyperscanning analysis using the HyPyP library. We will:
- Load epoch data for two participants.
- Construct a dyad (by combining the data into a single array).
- Compute a synchronization metric (circular correlation) using a connectivity analysis function.
- Visualize the resulting inter-brain synchrony connectivity matrix.

In [ ]:
import mne
import numpy as np
from collections import OrderedDict

# HyPyP modules for I/O, analyses, and visualization
import hypyp.io as io  # For loading and constructing dyads
import hypyp.analyses as analyses  # For computing synchronization metrics
import hypyp.prep as prep  # Preprocessing module (for ICA and other cleaning routines)
import hypyp.viz as viz  # For visualizing results

# Confirm successful import of libraries
print("Libraries imported successfully.")

## Loading the Data

We load the epoch files for two participants from:
- `./data/participant1-epo.fif`
- `./data/participant2-epo.fif`

Each file contains one epoch (a single trial) for one participant. After loading, we equalize the number of epochs between participants and print summaries for verification.

In [ ]:
# Load epochs for participant 1 and participant 2
epo1 = mne.read_epochs("./data/participant1-epo.fif", preload=True)
epo2 = mne.read_epochs("./data/participant2-epo.fif", preload=True)

# Equalize the number of epochs between participants to ensure consistent analysis
mne.epochs.equalize_epoch_counts([epo1, epo2])

# Print summaries to verify that the epochs have been loaded correctly
print("Participant 1 Epochs:")
print(epo1)
print("\nParticipant 2 Epochs:")
print(epo2)

## Preprocessing with ICA

Before computing connectivity, we perform additional preprocessing to remove artifacts such as eye blinks.  
Here we apply ICA using functions from `hypyp.prep`. Adjust parameters (e.g., method, number of components) as needed.

In [ ]:
# Compute ICA for each participant with 15 components
icas = prep.ICA_fit([
    epo1, epo2
],
    n_components=15,
    method='infomax',
    fit_params=dict(extended=True),
    random_state=42
)

# Select the relevant independent components for artefact rejection
cleaned_epochs_ICA = prep.ICA_choice_comp(icas, [epo1, epo2])
print('ICA correction completed.')

# Apply local AutoReject on the ICA-cleaned epochs
cleaned_epochs_AR, dic_AR = prep.AR_local(
    cleaned_epochs_ICA,
    strategy="union",
    threshold=50.0,
    verbose=True
)
print('AutoReject completed.')

# Assign cleaned epochs to individual participant variables
epo1_clean = cleaned_epochs_AR[0]
epo2_clean = cleaned_epochs_AR[1]
print('Preprocessed epochs for both participants are ready.')

# Update dyad with cleaned data for subsequent analysis
dyad_clean = [epo1_clean.get_data(copy=True), epo2_clean.get_data(copy=True)]

## Computing the Inter-Brain Synchrony (Circular Correlation)

In this section, we compute a synchronization metric using the circular correlation coefficient ("ccorr") rather than PLV. The steps are as follows:

1. **Determine Sampling Rate:**  
   We extract the sampling rate from one of the epochs.

2. **Define Frequency Bands:**  
   We define two frequency bands as an OrderedDict. Here, we focus on the "Alpha-Low" band for further analysis.

3. **Prepare Data:**  
   We combine the epochs from both participants into a single 4D array with shape *(2, n_epochs, n_channels, n_times)*.

4. **Compute Analytic Signal:**  
   The function `compute_freq_bands` filters the data and applies the Hilbert transform for each frequency band.

5. **Compute Connectivity:**  
   Using the `compute_sync` function with mode `'ccorr'`, we compute the inter-brain connectivity and then slice out the inter-brain connectivity matrix for the Alpha-Low band.

6. **Normalization:**  
   Finally, we compute a Z-score normalized connectivity matrix.

In [ ]:
# Extract the sampling rate from the epoch (assumes both participants share the same sfreq)
sampling_rate = epo1.info['sfreq']

# Define frequency bands as a dictionary (here two alpha sub-bands)
freq_bands = {
    'Alpha-Low': [7.5, 11],
    'Alpha-High': [11.5, 13]
}
# Convert to an OrderedDict to maintain the order
freq_bands = OrderedDict(freq_bands)

# Prepare data for connectivity analysis by combining both participants' epochs.
# The resulting data_inter array will have shape: (2, n_epochs, n_channels, n_times)
dyad_clean = np.array([epo1_clean.get_data(copy = True), epo2_clean.get_data(copy = True)])

# Compute the analytic signal in each frequency band using FIR filtering and Hilbert transform.
complex_signal = analyses.compute_freq_bands(
    dyad_clean,
    sampling_rate,
    freq_bands,
    filter_length=int(sampling_rate),  # Adjust filter length based on sampling rate
    l_trans_bandwidth=5.0,  # Reduced transition bandwidth for sharper filtering
    h_trans_bandwidth=5.0
)

# Compute connectivity using the circular correlation ('ccorr') metric and average across epochs.
result = analyses.compute_sync(complex_signal, mode='ccorr', epochs_average=True)

# Determine the number of channels per participant
n_ch = len(epo1_clean.info['ch_names'])

# Slice the connectivity matrix to extract inter-brain connectivity.
# The matrix 'result' has shape (n_freq, 2*n_channels, 2*n_channels).
# We slice to get connectivity values between channels of participant 1 (first n_ch)
# and participant 2 (last n_ch) for each frequency band.
alpha_low, alpha_high = result[:, 0:n_ch, n_ch:2*n_ch]

# For further analysis, choose the Alpha-Low band values.
values = alpha_low

# Compute a Z-score normalized connectivity matrix for improved comparability.
C = (values - np.mean(values[:])) / np.std(values[:])

## Visualizing the Results

We now visualize the computed inter-brain connectivity using both 2D and 3D representations.  
- The **2D topographic plot** helps identify regions with stronger inter-brain synchrony.
- The **3D visualization** provides a spatial representation of the connectivity.

The functions `viz.viz_2D_topomap_inter` and `viz.viz_3D_inter` handle the visualization.

In [ ]:
# Plot the 2D topographic map of the normalized connectivity matrix
viz.viz_2D_topomap_inter(epo1_clean, epo2_clean, C, threshold='auto', steps=10, lab=True)

In [ ]:
# Plot the 3D visualization of the inter-brain connectivity
viz.viz_3D_inter(epo1_clean, epo2_clean, C, threshold='auto', steps=10, lab=False)
print('3D inter-brain connectivity visualization completed.')

## Conclusion

In this notebook, we have:
- Loaded epoch data for two participants.
- Constructed a dyad by combining the data arrays.
- Computed a synchronization metric (circular correlation, "ccorr") to assess inter-brain synchrony across defined frequency bands.
- Visualized the resulting connectivity matrix using both 2D and 3D plots.

This foundational analysis prepares us for further hyperscanning investigations using HyPyP. In upcoming notebooks, we will explore more advanced preprocessing techniques, compare different synchronization metrics, and implement detailed statistical analyses.